In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import time
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

In [ ]:
df_weather = pd.read_csv('../data_file/hanoi_weather_history.csv')
df_weather.describe()


| Cột          | Mô tả                                                                                       |
|--------------|---------------------------------------------------------------------------------------------|
| `datetime`   | Thời gian ghi dữ liệu, dạng chuỗi (object), chứa thông tin về ngày và giờ.                 |
| `temp`       | Nhiệt độ thực tế (độ C), kiểu dữ liệu `float64`.                                            |
| `app_temp`   | Nhiệt độ cảm nhận (độ C), kiểu dữ liệu `float64`.                                           |
| `rh`         | Độ ẩm tương đối (%) của không khí, kiểu dữ liệu `int64`.                                     |
| `wind_spd`   | Tốc độ gió (m/s), kiểu dữ liệu `float64`.                                                   |
| `wind_dir`   | Hướng gió (độ từ 0 đến 360), kiểu dữ liệu `int64`.                                           |
| `pres`       | Áp suất khí quyển (hPa), kiểu dữ liệu `int64`.                                              |
| `vis`        | Tầm nhìn (m), kiểu dữ liệu `float64`.                                                      |
| `clouds`     | Tỷ lệ mây che phủ (%), kiểu dữ liệu `int64`.                                                |
| `precip`     | Lượng mưa (mm), kiểu dữ liệu `float64`.                                                     |
| `uv`         | Chỉ số UV (độ mạnh của tia cực tím), kiểu dữ liệu `float64`.                                |
| `dewpt`      | Nhiệt độ sương (độ C), kiểu dữ liệu `float64`.                                              |

In [ ]:
df_air = pd.read_csv('../data_file/hanoi_air_quality_history.csv')
df_air.describe()


| Cột        | Mô tả                                                                                       |
|------------|---------------------------------------------------------------------------------------------|
| `datetime` | Thời gian ghi dữ liệu, dạng chuỗi (object), chứa thông tin về ngày và giờ.                 |
| `aqi`      | Chỉ số chất lượng không khí (Air Quality Index), kiểu dữ liệu `int64`.                      |
| `pm25`     | Nồng độ bụi mịn PM2.5 (μg/m³), kiểu dữ liệu `float64`.                                      |
| `pm10`     | Nồng độ bụi PM10 (μg/m³), kiểu dữ liệu `float64`.                                          |
| `o3`       | Nồng độ Ozone (μg/m³), kiểu dữ liệu `float64`.                                              |
| `so2`      | Nồng độ Sulfur Dioxide (μg/m³), kiểu dữ liệu `float64`.                                    |
| `no2`      | Nồng độ Nitrogen Dioxide (μg/m³), kiểu dữ liệu `float64`.                                  |
| `co`       | Nồng độ Carbon Monoxide (μg/m³), kiểu dữ liệu `float64`.                                   |


### Merge dữ liệu từ hai DataFrame

Để có thể dự đoán PM2.5 dựa trên các yếu tố thời tiết, chúng ta cần kết hợp dữ liệu từ hai DataFrame lại với nhau theo cột thời gian.

In [ ]:
# Kiểm tra format của cột datetime trong cả hai DataFrame
print("Format datetime trong df_weather:")
print(df_weather['datetime'].head())
print("\nFormat datetime trong df_air:")
print(df_air['datetime'].head())

# Kiểm tra kiểu dữ liệu
print(f"\nKiểu dữ liệu datetime trong df_weather: {df_weather['datetime'].dtype}")
print(f"Kiểu dữ liệu datetime trong df_air: {df_air['datetime'].dtype}")

# Kiểm tra số lượng bản ghi
print(f"\nSố bản ghi df_weather: {len(df_weather)}")
print(f"Số bản ghi df_air: {len(df_air)}")

# Kiểm tra khoảng thời gian
print(f"\nKhoảng thời gian df_weather: từ {df_weather['datetime'].min()} đến {df_weather['datetime'].max()}")
print(f"Khoảng thời gian df_air: từ {df_air['datetime'].min()} đến {df_air['datetime'].max()}")

In [ ]:
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'], format='%Y-%m-%d:%H')
df_air['datetime'] = pd.to_datetime(df_air['datetime'], format='%Y-%m-%d:%H')

# Check định dạng sau chuyển đổi
print(f"df_weather datetime type: {df_weather['datetime'].dtype}")
print(f"df_air datetime type: {df_air['datetime'].dtype}")

In [ ]:
merged_df = pd.merge(df_weather, df_air, on='datetime', how='inner')

In [ ]:
print("\nThông tin DataFrame sau khi merge:")
print(f"Số bản ghi: {len(merged_df)}")
print(f"Số cột: {len(merged_df.columns)}")
print(f"Kích thước: {merged_df.shape}")

print(f"\nKhoảng thời gian dữ liệu: từ {merged_df['datetime'].min()} đến {merged_df['datetime'].max()}")

# Hiển thị thông tin về DataFrame đã merge
print("\nThông tin chi tiết DataFrame đã merge:")
merged_df.info()
merged_df.head()

In [ ]:
# 1. Thống kê missing value
print('=' * 80)
missing_count = merged_df.isna().sum()
missing_pct = (merged_df.isna().mean() * 100).round(2)
missing_summary = (
    pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
    .sort_values("missing_pct", ascending=False)
)
print("Tóm tắt missing values (chỉ hiển thị cột có thiếu):")
print(missing_summary[missing_summary.missing_count > 0])
print(f"\nTổng số hàng có ít nhất 1 giá trị thiếu: {merged_df.isna().any(axis=1).sum()} / {len(merged_df)}")

# 2. Thống kê trùng lặp
print('=' * 80)
dup_rows = merged_df.duplicated().sum()
print(f"Số dòng trùng lặp tuyệt đối: {dup_rows}")

# 3. Thống kê Outliers theo IQR
print('=' * 80)
numeric_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()

def iqr_bounds(s: pd.Series):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        return None, None
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_stats = []
for col in numeric_cols:
    col_series = merged_df[col].dropna()
    low, up = iqr_bounds(col_series)
    if low is None:
        outlier_stats.append((col, 0, 0.0, np.nan, np.nan))
        continue
    mask = (merged_df[col] < low) | (merged_df[col] > up)
    cnt = int(mask.sum())
    pct = round(100 * cnt / len(merged_df), 2)
    outlier_stats.append((col, cnt, pct, low, up))

outlier_summary = pd.DataFrame(
    outlier_stats, columns=["feature", "outlier_count", "outlier_pct", "lower_bound", "upper_bound"]
).sort_values("outlier_pct", ascending=False)

print("\nTop 10 biến có outliers nhiều nhất:")
print(outlier_summary.head(10))

any_outlier_mask = pd.Series(False, index=merged_df.index)
for col in numeric_cols:
    low, up = iqr_bounds(merged_df[col].dropna())
    if low is None:
        continue
    any_outlier_mask |= (merged_df[col] < low) | (merged_df[col] > up)

print(f"\nSố hàng có ít nhất 1 outlier: {any_outlier_mask.sum()} / {len(merged_df)}")

In [ ]:
# Xử lý biến hướng gió
# Chuẩn hoá 360 -> 0
merged_df['wind_dir'] = merged_df['wind_dir'] % 360
# Tạo encoding dạng vòng tròn
merged_df['wind_dir_rad'] = np.deg2rad(merged_df['wind_dir'])
merged_df['wind_dir_sin'] = np.sin(merged_df['wind_dir_rad'])
merged_df['wind_dir_cos'] = np.cos(merged_df['wind_dir_rad'])

# Bỏ wind_dir và wind_dir_rad sau khi đã tạo xong encoding
merged_df.drop(columns=['wind_dir', 'wind_dir_rad'], inplace=True)

In [ ]:
merged_df.select_dtypes(include=[np.number]).describe()